In [1]:
# Cell 1 - Imports & Load
import json
import re
import shutil
from collections import defaultdict
from pathlib import Path

INPUT_PATH  = "skill_alias_v6.json"
OUTPUT_PATH = "skill_alias_v7.json"

with open(INPUT_PATH, encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data):,} entries from {INPUT_PATH}")

Loaded 40,505 entries from skill_alias_v6.json


In [2]:
# Cell 2 - Rules & Whitelist

# Canonical được giữ lại dù bị flag bởi rule
CANONICAL_WHITELIST = {
    # Abbreviations hợp lệ 2 ký tự
    "AI", "AR", "VR", "UI", "UX", "C#", "Go", "R",
    # .NET ecosystem - starts with dot nhưng là tech thật
    ".NET",
    # Digit-prefix nhưng là methodology/tool thật
    "3ds Max", "5why", "5 whys", "7 wastes", "7 quality tools",
    "7 security operations", "7qc tools", "8d scar",
    "5 quality tools", "5-axis cnc milling", "5m1e", "5why",
    "4-point inspection system", "4m1e", "6signma",
    "3d printing", "3d printer", "3d studio max", "3d mockup",
    "3d solid modeling", "3d electrical design", "3d coordination",
    "360-degree feedback",
}

def is_noise_canonical(canonical: str) -> bool:
    """
    Trả về True nếu canonical bị coi là noise → đánh SKIP.
    Kiểm tra whitelist trước, sau đó áp rules.
    """
    c = canonical.strip()

    # Luôn giữ lại nếu trong whitelist
    if c in CANONICAL_WHITELIST:
        return False

    # R1: quá ngắn (<=2 ký tự), loại trừ known abbreviations
    if len(c) <= 2:
        return True

    # R2: bắt đầu bằng ký tự đặc biệt (không phải chữ cái hoặc digit hợp lệ)
    if re.match(r'^[\.\-\&\#\*\/\(\)\[\]_\+\~\,\;\:\!\?]', c):
        return True

    # R3: CJK fragment 1-2 ký tự (Trung/Nhật/Hàn đơn lẻ)
    if re.match(r'^[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]{1,2}$', c):
        return True

    # R4: bắt đầu bằng số rồi space → sentence fragment (vd: "2022 về kcn")
    #     Ngoại lệ: digit ngay sau là chữ hoa/D/d → acronym như "3D", "2D", "4G"
    if re.match(r'^\d+\s', c) and not re.match(r'^\d+[A-Za-z]', c):
        return True

    # R5: lowercase đầu + dài > 20 ký tự → sentence bị NER bắt lầm
    # Chú ý: tiếng Việt có ký tự đặc biệt như Đ, Ă... không phải a-z
    # Rule này chỉ áp cho ký tự đặc biệt tiếng Việt thường bắt đầu bằng chữ thường
    if re.match(r'^[a-zđăâêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]', c) and len(c) > 20:
        return True

    return False

# Test nhanh
test_cases = [
    ("AI", False), (".NET", False), ("3ds Max", False),
    ("360-degree feedback", False), ("Go", False),
    ("De", True), ("& product", True), ("2022 về kcn", True),
    ("Đánh giá hiệu quả chiến dịch marketing", True),
    ("中", True), ("Python", False), ("Tiếng Anh", False),
]
print("Rule tests:")
for c, expected in test_cases:
    result = is_noise_canonical(c)
    status = "✅" if result == expected else "❌"
    print(f"  {status} {repr(c):45s} noise={result} (expected={expected})")

Rule tests:
  ✅ 'AI'                                          noise=False (expected=False)
  ✅ '.NET'                                        noise=False (expected=False)
  ✅ '3ds Max'                                     noise=False (expected=False)
  ✅ '360-degree feedback'                         noise=False (expected=False)
  ✅ 'Go'                                          noise=False (expected=False)
  ✅ 'De'                                          noise=True (expected=True)
  ✅ '& product'                                   noise=True (expected=True)
  ✅ '2022 về kcn'                                 noise=True (expected=True)
  ❌ 'Đánh giá hiệu quả chiến dịch marketing'      noise=False (expected=True)
  ✅ '中'                                           noise=True (expected=True)
  ✅ 'Python'                                      noise=False (expected=False)
  ✅ 'Tiếng Anh'                                   noise=False (expected=False)


In [3]:
# Cell 3 - Apply cleaning

skipped_before = sum(1 for v in data.values() if v["canonical"] == "__SKIP__")

newly_skipped_canonicals = set()
cleaned_data = {}

for key, val in data.items():
    canonical = val["canonical"]
    if canonical != "__SKIP__" and is_noise_canonical(canonical):
        newly_skipped_canonicals.add(canonical)
        cleaned_data[key] = {"canonical": "__SKIP__", "type": val["type"]}
    else:
        cleaned_data[key] = val

skipped_after = sum(1 for v in cleaned_data.values() if v["canonical"] == "__SKIP__")
newly_skipped_entries = skipped_after - skipped_before

print(f"Entries trước khi clean  : {len(data):,}")
print(f"__SKIP__ cũ (v6)         : {skipped_before:,}")
print(f"Canonical mới bị SKIP    : {len(newly_skipped_canonicals):,} canonicals")
print(f"Entries mới bị SKIP      : {newly_skipped_entries:,} entries")
print(f"__SKIP__ sau clean (v7)  : {skipped_after:,}")
print(f"Entries còn lại có nghĩa : {len(cleaned_data) - skipped_after:,}")

Entries trước khi clean  : 40,505
__SKIP__ cũ (v6)         : 763
Canonical mới bị SKIP    : 86 canonicals
Entries mới bị SKIP      : 128 entries
__SKIP__ sau clean (v7)  : 891
Entries còn lại có nghĩa : 39,614


In [4]:
# Cell 4 - Review newly skipped canonicals

print(f"=== {len(newly_skipped_canonicals)} canonicals mới bị đánh __SKIP__ ===\n")

# Nhóm theo lý do để dễ review
groups = {
    "too_short (<=2 chars)": [],
    "starts_special": [],
    "cjk_fragment": [],
    "digit_sentence": [],
    "long_lowercase_fragment": [],
}

for c in sorted(newly_skipped_canonicals):
    if len(c.strip()) <= 2 and c not in CANONICAL_WHITELIST:
        groups["too_short (<=2 chars)"].append(c)
    elif re.match(r'^[\.\-\&\#\*\/\(\)\[\]_\+\~\,\;\:\!\?]', c):
        groups["starts_special"].append(c)
    elif re.match(r'^[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]{1,2}$', c):
        groups["cjk_fragment"].append(c)
    elif re.match(r'^\d+\s', c) and not re.match(r'^\d+[A-Za-z]', c):
        groups["digit_sentence"].append(c)
    else:
        groups["long_lowercase_fragment"].append(c)

for group_name, items in groups.items():
    if items:
        print(f"--- {group_name} ({len(items)}) ---")
        for c in items:
            print(f"  {repr(c)}")
        print()

=== 86 canonicals mới bị đánh __SKIP__ ===

--- too_short (<=2 chars) (64) ---
  '3a'
  'A'
  'Al'
  'Am'
  'An'
  'Cu'
  'Cy'
  'Da'
  'De'
  'Di'
  'Eb'
  'Ed'
  'Er'
  'Eu'
  'Ev'
  'Ex'
  'Fi'
  'Id'
  'Im'
  'Ma'
  'Mo'
  'Oa'
  'Oh'
  'Ow'
  'Pu'
  'Qe'
  'Se'
  'So'
  'Us'
  'Vi'
  'Vẽ'
  'Án'
  'Ô'
  'Ả'
  'Ẩ'
  'Ế'
  '中'
  '交期'
  '交流'
  '光伏'
  '公关'
  '協調'
  '听说'
  '商业'
  '外联'
  '工厂'
  '工單'
  '应变'
  '授课'
  '整合'
  '文档'
  '普通'
  '検査'
  '溝通'
  '点赞'
  '管理'
  '粘贴'
  '能'
  '英'
  '英語'
  '薪酬'
  '转化'
  '锦浪'
  '영어'

--- starts_special (12) ---
  '& d strategie'
  '& product'
  '& sustainability'
  '- english'
  '- phát triển'
  '- thiết lập và vận hành môi trường sử dụng public cloud'
  '- tiếng anh'
  '- triển khai bán hàng'
  '- triển khai các quy trình tự động hóa'
  '. net automation'
  '. net core mvc'
  '_SKIP__'

--- digit_sentence (10) ---
  '2 communication'
  '2 welding'
  '2015 quality management systems six sigma'
  '2022 về kcn'
  '3 chứng chỉ bảo mẫu'
  '360 campaign deploy

In [5]:
# Cell 5 - Manual rescue (chạy sau khi review Cell 4)
# Thêm bất kỳ canonical nào bị SKIP nhầm vào đây rồi re-run Cell 3

EXTRA_WHITELIST = {
    # Ví dụ nếu bạn muốn giữ lại:
    # "360 marketing management",
    # "2d animation",
}

CANONICAL_WHITELIST.update(EXTRA_WHITELIST)

# Re-run lại Cell 3 sau khi update whitelist
print("Whitelist updated. Re-run Cell 3 để áp dụng.")
print(f"Current whitelist size: {len(CANONICAL_WHITELIST)}")

Whitelist updated. Re-run Cell 3 để áp dụng.
Current whitelist size: 31


In [6]:
# Cell 6 - Export skill_alias_v7.json

# Backup v6 phòng hờ
shutil.copy(INPUT_PATH, INPUT_PATH.replace(".json", "_backup.json"))

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print(f"Saved {OUTPUT_PATH}")
print(f"File size: {Path(OUTPUT_PATH).stat().st_size / 1024 / 1024:.2f} MB")

# Sanity check
with open(OUTPUT_PATH, encoding="utf-8") as f:
    verify = json.load(f)
assert len(verify) == len(cleaned_data), "Entry count mismatch!"
print(f"Verified: {len(verify):,} entries OK")

Saved skill_alias_v7.json
File size: 3.84 MB
Verified: 40,505 entries OK
